# Baseline tháng gần nhất + lịch sử tích lũy cũ

Dự đoán `y_any` cho tháng M từ tháng M−1 và lịch sử trước M−1. Đọc [đặc tả](../docs/propensity_baseline_design.md) trước khi chạy. Notebook điều phối module `propensity_baseline`; mã triển khai nằm trong `src/propensity_baseline/`.

Chạy từ repo đã cài `pip install -e ".[baseline]"`. Calendar và tune từ chối ghi đè output. Chỉ bước test có cờ mở riêng. Chưa chạy notebook này trên toàn bộ MBD-mini.

In [ ]:
from pathlib import Path
import json
from propensity_baseline.config import load_paths, load_baseline_config
from propensity_baseline.data import build_calendar
from propensity_baseline.runner import tune, fit_final, test_final, locked_run, source_contract

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Chạy notebook trong repository"
calendar_paths = load_paths(ROOT / "configs/propensity_calendar.json")
config = load_baseline_config(ROOT / "configs/propensity_baseline.json")
print(calendar_paths)
print(config.to_dict())

## 1. Tạo feature theo tháng lịch

Đọc cache chuẩn hóa của notebook 03 ở chế độ read-only. `recent=[đầu M−1, đầu M)`, `older<trước đầu M−1`. Xuất schema v3 vào output mới. Bước này có thể tốn thời gian; không fit mô hình.

In [ ]:
calendar_manifest = calendar_paths["output_dir"] / "manifest.json"
if calendar_manifest.exists():
    prepared = json.loads(calendar_manifest.read_text(encoding="utf-8"))
    assert prepared.get("status") == "ready", "Output calendar chưa ready; kiểm tra run trước hoặc dùng output mới"
    source_contract(config)
    print("Dùng feature calendar ready hiện có:", calendar_manifest)
else:
    calendar_manifest = build_calendar(**calendar_paths)
    print(calendar_manifest)

## 2. Baseline đa nguồn và chọn cấu hình

Cấu hình mặc định dùng cả trx, geo và dialog, so sánh recent với recent+older. Dialog dùng metadata hoạt động, chưa dùng nội dung embedding. Chỉ đọc train/validation; không đọc tháng calibration hoặc test. Primary: Recall@1%.

In [ ]:
selection_path = config.output_dir / "selection.json"
run_manifest = config.output_dir / "manifest.json"
if run_manifest.exists():
    run_state = json.loads(run_manifest.read_text(encoding="utf-8"))
    assert run_state.get("status") in ["tuned", "fitted", "tested"], "Run đang chạy hoặc lỗi; không ghi đè"
    locked_run(config, run_state["status"])
    print("Dùng selection đã khóa:", selection_path)
else:
    selection_path = tune(config)
selection = json.loads(selection_path.read_text(encoding="utf-8"))
print(json.dumps({k:v for k,v in selection.items() if k not in ["features","protocol"]}, ensure_ascii=False, indent=2))

## 3. Đọc kết quả validation trước khi khóa mô hình

Kiểm tra `validation_metrics.json`, `paired_comparison.json` và `ablation_comparisons.json`. Khoảng tin cậy validation mang tính thăm dò vì dùng cùng tháng để chọn cấu hình. Nếu muốn đổi thiết kế, tạo run mới; không sửa selection đã khóa.

In [ ]:
comparison = json.loads((config.output_dir / "ablation_comparisons.json").read_text(encoding="utf-8"))
print(json.dumps(comparison, ensure_ascii=False, indent=2))

## So sánh với rule mua tháng trước

Rule `y_hat(M)=y_any(M−1)`; nhãn trước thiếu/chưa sẵn giữ unknown. So trên cùng cohort và cùng top-K; báo thêm ngân sách bằng số người mua tháng trước. Rule là quyết định 0/1, không phải xác suất đã hiệu chỉnh.

In [ ]:
from propensity_baseline.persistence import compare_rule
import pandas as pd
rule_directory = config.output_dir / "rule_validation"
if rule_directory.exists():
    assert json.loads((rule_directory / "manifest.json").read_text(encoding="utf-8"))["status"] == "ready"
else:
    rule_directory = compare_rule(config)
display(pd.read_csv(rule_directory / "monthly_comparison.csv"))

## 4. Fit cuối và calibration

Fit base model trên 02–11/2022; tháng 12 chỉ fit sigmoid calibrator đã quy định. Base model giữ nguyên sau calibration. Chưa mở nhãn test.

In [ ]:
run_state = json.loads((config.output_dir / "manifest.json").read_text(encoding="utf-8"))
if run_state["status"] in ["fitted", "tested"]:
    locked_run(config, run_state["status"])
    final_directory = config.output_dir / "final"
    print("Dùng mô hình cuối đã fit:", final_directory)
else:
    final_directory = fit_final(config)
    print(final_directory)

## 5. Test khóa

Chỉ đổi cờ sau khi toàn bộ thiết kế và ứng viên đã chốt. Kết quả tháng 01/2023 không được dùng để đổi cấu hình hoặc chọn mô hình. Luồng từ chối mở test lần hai trong cùng run.

In [ ]:
OPEN_FINAL_TEST = False
if OPEN_FINAL_TEST:
    print(test_final(config))
    print(compare_rule(config, stage="test"))
else:
    print("Test chưa được mở. Chốt thiết kế và mô hình trước khi đổi OPEN_FINAL_TEST.")